[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/07_end_to_end.ipynb)

# End-to-end synthetic experiment

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: The pieces in notebooks 00–06 assemble into one callable,
`run_validation_experiment`, also exposed as `multilogue-validate`. It uses only
synthetic units and FakeClient — safe for CI and Colab without keys — and writes
tables, a multiverse decomposition, and an evidence pack. Use this notebook as a
smoke test after installing multilogue, then return to earlier notebooks when you
want to change one aspect at a time.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

A full Address path is a sequence of explicit stages: extract, cluster, weight,
multiverse, validate. The experiment helper wires them with defaults so that
“does the install work?” is one cell, not ten. It is not a substitute for the
methodological walkthroughs — it is the integration test you can also read.


In [ ]:
from IPython.display import display
from pathlib import Path
import pandas as pd
from multilogue.experiment import run_validation_experiment
from multilogue import viz


In [ ]:
out = Path("outputs/notebook_e2e")
summary = run_validation_experiment(
    out, n_docs_per_group=3, units_per_doc=3, k=4, quiet=True,
)
display(summary)
tables = out / "tables"
for name in ["yield.csv", "group_compare_lenses.csv", "variance_shares.csv"]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head())


In [ ]:
shares = pd.read_csv(tables / "variance_shares.csv")
viz.variance_share_bars(shares, path=FIG / "07_variance_shares.png")
print("Evidence pack:", out / "evidence_pack.html")


# Outlook

- Soft PPCA lenses and document EMD: notebook **08**.
- Formats-style Address (format as group, format-gap $\ell$): notebook **09**.
- Swap `load("synthetic")` for a public corpus in Access when you want real civic
  text; replace FakeClient with an OpenAI or LiteLLM client for real extraction
  (checkpoints still apply).
